In [123]:

from datetime import datetime, timedelta
import math
import pandas as pd
import numpy as np
import requests


In [124]:
def obtener_ventas(token, fecha_inicio, fecha_fin):
    url = "https://back-middleware.eurosupermercados.com/euro_nous_thirdparties/sales/get_sales/"
    headers = {"Authorization": f"key {token}", "Content-Type": "application/json"}

    fecha_inicio = pd.to_datetime(fecha_inicio)
    fecha_fin = pd.to_datetime(fecha_fin)

    ventas_totales = []
    dias_faltantes = []

    fecha_actual = fecha_inicio
    while fecha_actual < fecha_fin:

        fecha_str = fecha_actual.strftime('%d/%m/%Y')
        params = {"date__date_sale": fecha_str}

        print(f"Consultando ventas del día: {fecha_str}")
        try:
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status()
            data = response.json().get("data", [])
            ventas_totales.extend(data)

        except Exception as e:
            print(f"Error en el día {fecha_str}")
            dias_faltantes.extend(fecha_str)

        fecha_actual += timedelta(days=1)

    return ventas_totales, dias_faltantes

def obtener_token(username, password):
    url = "https://back-middleware.eurosupermercados.com/api-auth/"
    payload = {"username": username, "password": password}
    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()["key"]

In [125]:


df_procesar = pd.read_csv(
    "../data/raw/ventas_completo.csv",
    usecols=['identification_doct', 'product', 'date_sale'])

# Renombrar y filtrar IDs de cliente
df_procesar['client'] = df_procesar['identification_doct'].astype(str).str.strip()
mask_digits = df_procesar["client"].str.isdigit().fillna(False)
mask_zero = ~df_procesar["client"].str.startswith("0", na=False)
df_procesar = df_procesar[mask_digits & mask_zero].copy()
print(f"Filas tras filtrar id_client no numéricos/cero inicial: {len(df_procesar)}")

# Limpiar fechas, productos domicilio
df_procesar['product'] = df_procesar['product'].astype(str).str.strip()
df_procesar['date_sale'] = pd.to_datetime(df_procesar['date_sale'])
df_procesar = df_procesar.dropna(subset=['date_sale'])
df_procesar['date_sale'] = df_procesar['date_sale'].dt.normalize()


productos = pd.read_csv(
    f"../data/raw/productos.csv",
    dtype={"codigo_unico": str}
).rename(columns={"codigo_unico": "product"})
productos = productos.loc[:, ["product", "description", "brand", "category"]]
productos = productos.drop_duplicates("product")
productos['product'] = productos['product'].str.strip()
df_procesar = pd.merge(df_procesar, productos, on="product")
print(f"Datos cargados Filas: {len(df_procesar)}")


predictions = pd.read_csv("../predictions/preditions.csv", converters={"client": str})


/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_75374/1501608320.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_procesar = pd.read_csv(


Filas tras filtrar id_client no numéricos/cero inicial: 18407577
Datos cargados Filas: 18385402


In [126]:
LAST_MONTH = 1
TOP_PCT = 0.75
EXCLUIR = [
    'BOLSA PLASTICA',
    'BOLSA RECUPERADA',
    'MENU EMPLEADOS EURO',
]

In [158]:

print(f"Filtrando clientes con compras en los últimos {LAST_MONTH} meses...")
last_months_ago = datetime.now().date() - timedelta(days=LAST_MONTH*30)
recent_purchases = df_procesar[(df_procesar["date_sale"].dt.date >= last_months_ago)]
recent_purchases = recent_purchases[~recent_purchases["description"].isin(EXCLUIR)]
print(f"Compras en los últimos {LAST_MONTH} meses: {len(recent_purchases)}")
hist_recent_client = recent_purchases[recent_purchases["client"].isin(predictions["client"])]
print(f"Clientes predichos con compras en los últimos {LAST_MONTH} meses: {hist_recent_client["client"].nunique()}")
hist_recent_client["date_sale"].min()

Filtrando clientes con compras en los últimos 1 meses...
Compras en los últimos 1 meses: 2561192
Clientes predichos con compras en los últimos 1 meses: 419


Timestamp('2025-04-17 00:00:00')

In [159]:
# Conteo de compras por cliente-producto
cant_compras_client = (
    hist_recent_client
    .groupby(["client", "description"])
    .size()                               # cuenta filas
    .rename("cant_compras")
    .reset_index()
)
#Ordenar de mayor a menor para cada cliente
cant_compras_client = (
    cant_compras_client
    .sort_values(["client", "cant_compras"], ascending=[True, False])
)

def top_pct(df):
    k = math.ceil(len(df) * TOP_PCT)
    return df.head(k)

top_products = (
    cant_compras_client
      .groupby("client", group_keys=False)
      .apply(top_pct)
      .reset_index(drop=True)
)

In [160]:
top_summary = (
    top_products
    .groupby('client')
    .agg(
        productos_comunes=('description', list), #lambda x: ', '.join(x)
        cantidad_productos=('description', 'count')
    )
    .reset_index()
)
top_summary

,client,productos_comunes,cantidad_productos
0,1000192661,"[BUNUELO EURO 55 gr, PONY MALTA PET x 330ML, ...",27
1,1000410009,"[BRETAÑA FRIOPACK SUPER *300ML, BEBIDA ENERGIZ...",47
2,1000440285,"[GASEOSA COCACOLA *1.5L, COCACOLA x 600ML, G...",31
3,1000445397,"[ACOMPAÑAMIENTO COMIDA PPAL, BEBIDA ENERGIZAN ...",18
4,1000534537,"[AGUA LIMA/LIMON HOJAS MENTA ZEN*540ML, MR TEA...",72
...,...,...,...
414,98544040,"[LECHE COLANTA SEMIDESCREM 1000ml, CIGARRILLO ...",25
415,98552891,"[AGUA POTABLE TRATADA EUROMAX *1000ML, TAMPICO...",61
416,98558712,"[AREPA TELA BLANCA CASERA SARY X10 UNDS, QUESI...",20
417,98630133,"[AGUA BRISA LIMA LIMON PET*1500ML, AGUA H2O LI...",19


In [161]:
predictions_with_precom = pd.merge(
    predictions.loc[:, ["client", "prob"]],
    top_summary,
    on="client",
    how="left")

In [162]:
predictions_with_precom

,client,prob,productos_comunes,cantidad_productos
0,70569726,0.7422,"[CIGARRILLO CHESTERFIRLD BLUE BOX x 20UND, CIG...",81
1,800180330,0.7422,"[PLATANO, PULPA EUROMAX CONGELADA DE MANGO X ...",158
2,1235245925,0.7303,"[PECHUGA BUCANERO CAMPESINA REFRIGERADA, ARROZ...",66
3,901569974,0.7303,"[COSTILLA DE RES.QB, TRANSPORTE DOMICILIO, HUE...",115
4,1062404839,0.7181,"[SOPA DEL DIA X 300GR, MR TEA LIMON PET x 500...",24
...,...,...,...,...
414,1036619176,0.5005,"[ENSALADA FRESCA PORCION 50 GR, PAN MANTEQUILL...",21
415,1037617433,0.5005,"[BEBIDA HIDRATANTE MARACUYA TAO*600ML, AGUA PO...",21
416,43720824,0.5003,"[PIERNA PULPA.QB, PONY MALTA MINI PET x 200ML...",30
417,1038820652,0.5003,"[JUGO NATURAL X 12 OZ MENU, BEBIDA CON GAS ACQ...",21


In [163]:
hist_recent_client[hist_recent_client["client"] == "7797719"]["description"].nunique()

62

In [164]:
predictions_with_precom

,client,prob,productos_comunes,cantidad_productos
0,70569726,0.7422,"[CIGARRILLO CHESTERFIRLD BLUE BOX x 20UND, CIG...",81
1,800180330,0.7422,"[PLATANO, PULPA EUROMAX CONGELADA DE MANGO X ...",158
2,1235245925,0.7303,"[PECHUGA BUCANERO CAMPESINA REFRIGERADA, ARROZ...",66
3,901569974,0.7303,"[COSTILLA DE RES.QB, TRANSPORTE DOMICILIO, HUE...",115
4,1062404839,0.7181,"[SOPA DEL DIA X 300GR, MR TEA LIMON PET x 500...",24
...,...,...,...,...
414,1036619176,0.5005,"[ENSALADA FRESCA PORCION 50 GR, PAN MANTEQUILL...",21
415,1037617433,0.5005,"[BEBIDA HIDRATANTE MARACUYA TAO*600ML, AGUA PO...",21
416,43720824,0.5003,"[PIERNA PULPA.QB, PONY MALTA MINI PET x 200ML...",30
417,1038820652,0.5003,"[JUGO NATURAL X 12 OZ MENU, BEBIDA CON GAS ACQ...",21


## Contraste con las compras ocurridas

In [165]:
token = obtener_token()
nuevas_ventas, _ = obtener_ventas(token, "2025-05-15", "2025-05-16")

Consultando ventas del día: 15/05/2025


In [166]:
df_nuevas_ventas = pd.DataFrame(nuevas_ventas)
ventas_explotado = df_nuevas_ventas.explode('invoice_details').reset_index(drop=True)
invoice_df = pd.json_normalize(ventas_explotado['invoice_details'])
ventas_final = pd.concat([ventas_explotado.drop(columns=['invoice_details']).reset_index(drop=True),
                          invoice_df.reset_index(drop=True)], axis=1)
ventas_final['date_sale'] = pd.to_datetime(ventas_final['date_sale'])
ventas_final["client"] = ventas_final["identification_doct"].astype(str).str.strip()
ventas_final["product"] = ventas_final["product"].astype(str).str.strip()
ventas_final = pd.merge(ventas_final, productos, on="product")
ventas_final = ventas_final[ventas_final["client"].isin(predictions["client"])]

In [167]:
compras = ventas_final.groupby("client").agg(
    productos_comunes_real=('description', list), #lambda x: ', '.join(x)
    cantidad_productos_real=('description', 'count')).reset_index()

In [168]:
rev = pd.merge(predictions_with_precom, compras, on="client", how='inner')

In [169]:
rev["productos_interseccion"] = rev.apply(
    lambda r: set(r["productos_comunes"]) & set(r["productos_comunes_real"]),
    axis=1
)
rev["cant_productos_interseccion"] = rev.apply(
    lambda r: len(set(r["productos_comunes"]) & set(r["productos_comunes_real"])),
    axis=1
)

In [170]:
rev

,client,prob,productos_comunes,cantidad_productos,productos_comunes_real,cantidad_productos_real,productos_interseccion,cant_productos_interseccion
0,70569726,0.7422,"[CIGARRILLO CHESTERFIRLD BLUE BOX x 20UND, CIG...",81,"[PANELA SAN JOAQUIN FRACCION x 960GR, PLATANO,...",15,"{PAPA CAPIRA OFERTA, PERIODICO Q HUBO LUNES A ...",14
1,800180330,0.7422,"[PLATANO, PULPA EUROMAX CONGELADA DE MANGO X ...",158,"[PLATANO, PLATANO, AJO MALLA, CEBOLLA JUNCA PE...",13,"{CEBOLLA JUNCA PELADA UND, AREPAS EUROMAX BLAN...",9
2,1235245925,0.7303,"[PECHUGA BUCANERO CAMPESINA REFRIGERADA, ARROZ...",66,"[PECHUGA BUCANERO CAMPESINA REFRIGERADA, PIERN...",9,"{SALSA SOYA PET ADEREZOS x 1050GR, PIERNA PUL...",8
3,901569974,0.7303,"[COSTILLA DE RES.QB, TRANSPORTE DOMICILIO, HUE...",115,"[DETER. POLVO EUROMAX BICARBONATO X1000GR, COS...",31,"{BLANQUEADOR DESINFECTANTE X 3800ML EURO, PANE...",20
4,1062404839,0.7181,"[SOPA DEL DIA X 300GR, MR TEA LIMON PET x 500...",24,"[AGUACATE HASS, ATUN EN AGUA VANCAMPS x 160GR...",6,"{AGUACATE HASS, COCACOLA FLE x I x 400ML, MR...",4
...,...,...,...,...,...,...,...,...
304,1006868713,0.5005,"[BANANO CRIOLLO, GOMAS TRULULU AROS X 70 GR, G...",19,"[PECHUGA BUCANERO CAMPESINA REFRIGERADA, GASEO...",4,"{GASEOSA COCACOLA *1.5L, AREPA CHOCOLO RELL D...",2
305,1036398633,0.5005,"[BUNUELO EURO 55 gr, JUGO NATURAL X 12 OZ MENU...",12,"[HALLS EXTRA FUERTE LYPTUS BARRA X 25 GR, EMPA...",3,"{EMPANADA PAPA Y CARNE *60GR, PASTEL CON POLLO...",3
306,1036619176,0.5005,"[ENSALADA FRESCA PORCION 50 GR, PAN MANTEQUILL...",21,"[PROM HOJALDRE X 3, CHORIZO MIXTO QUALITY BEEF...",6,"{CHORIZO BRASILERO QUALITY BEEF x 500GR, ENSAL...",5
307,1037617433,0.5005,"[BEBIDA HIDRATANTE MARACUYA TAO*600ML, AGUA PO...",21,"[CHOCOLATINA JET WAFER VAINILLA x 22 GRS, BEBI...",6,"{CHOCOLATINA JET WAFER VAINILLA x 22 GRS, GATO...",3


In [171]:
rev[rev["cant_productos_interseccion"]<=0]

,client,prob,productos_comunes,cantidad_productos,productos_comunes_real,cantidad_productos_real,productos_interseccion,cant_productos_interseccion
27,901607201,0.6727,"[FILETE DE PECHUGA CAMPESINA, CALDO ESPECIAS ...",62,[YOGURT SKETOS PARFAIT FRUTOS ROJOS*150GR],1,{},0
63,1020479172,0.6232,"[EMPANADA DE CARNE x 100GR, GASEOSA PEPSI PET...",27,[UVA POSTOBON PET x 400 ML],1,{},0
101,1036631069,0.5993,"[ESPRESSO NESPRESSO X 3 OZ, SOPA DEL DIA X 300...",28,[MENU EMPLEADOS EURO],1,{},0
107,1148946241,0.5941,"[AGUA POTABLE CON GAS EUROMAX *600 ML, CABEZA ...",33,[PROMOCION HOJALDRES STDOS X 4 UDS],1,{},0
123,33309158,0.5845,"[AGUACATE HASS, AGUACATE INJERTO, BUNUELO EURO...",29,[MENU EMPLEADOS EURO],1,{},0
143,1003337090,0.5712,"[BEB ENERGIZAN SPEED MAX BLUE LATA *310ML, BEB...",17,[REFRESCO HIT FRUTAS TROPICALES *500ML],1,{},0
154,1001369027,0.5627,"[BUNUELO EURO 55 gr, COCACOLA x 600ML, TRONQU...",23,"[MENU EMPLEADOS EURO, VINO FIELO TINTO MERLOT ...",2,{},0
252,1038646759,0.5165,"[ENSALADA FRESCA PORCION 50 GR, PASTEL DE AREQ...",58,[SOPA DEL DIA X 300GR],1,{},0
268,1066741729,0.5090,"[BUNUELO EURO 55 gr, EMPANADA DE CARNE x 100G...",21,[PAN REY AL HORNO x 400GR],1,{},0
283,8355906,0.5074,"[JUGO NATURAL X 12 OZ MENU, CIGARRILLOS L&M PU...",31,[MENU EMPLEADOS EURO],1,{},0
